In [5]:
import pandas as pd

In [6]:
df = pd.read_csv("IMDB Dataset.csv")

In [7]:
df.shape

(50000, 2)

In [8]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [9]:
df.drop_duplicates(inplace=True)

In [10]:
df.shape

(49582, 2)

# Pre Processing

### 1. Coverting test to lowercase

In [11]:
df["review"] = df["review"].str.lower()

### 2. Removing the URLs

In [12]:
import re

def remove_urls(text):
    text = re.sub(r"http\S+","",text)
    return text

df["review"] = df["review"].apply(remove_urls)

### 3. Removing punctuations

In [13]:
def remove_punctuations(text):
    text = re.sub(r"^[A-Za-z0-9\s+]","",text)
    return text
df["review"] = df["review"].apply(remove_punctuations)

### 4. Removing HTML

In [14]:
def remove_HTML(text):
    text = re.sub(r"<.*?>","",text)
    return text
df["review"] = df["review"].apply(remove_HTML)

### 5. Removing the Stopwords

In [15]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to C:\Users\Sagar
[nltk_data]     Panwar\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to C:\Users\Sagar
[nltk_data]     Panwar\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\Sagar
[nltk_data]     Panwar\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [16]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text = text.replace(word,"")

    return text
df["review"] = df["review"].apply(remove_stopwords)

### 6. Stemming

In [17]:
from nltk.stem import PorterStemmer

def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)
        
    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

### 7. Encoding

In [18]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

### 8. Vectorization

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)

X = tf.fit_transform(df["review"])
y = df["sentiment"]

## Dataset and DataLoader

In [20]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.2,random_state=42
)

In [21]:
import torch
from torch.utils.data import TensorDataset, DataLoader

In [22]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [23]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

C:\Users\Sagar Panwar\AppData\Local\Temp\ipykernel_28944\2931448922.py:3: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  torch.from_numpy(y_train.values).float()


In [24]:
train_loader = DataLoader(train_set,batch_size=64,shuffle=True)
test_loader = DataLoader(test_set,batch_size=64,shuffle=True)

### Build RNN

In [25]:
import torch.nn as nn
import torch.optim as optim

In [26]:
class RNN(nn.Module):
    def __init__(self,input_size,hidden_size=128,num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        ## RNN layer
        self.rnn = nn.RNN(input_size,hidden_size,num_layers, batch_first=True)
        
        # fully connected layer
        self.fc = nn.Linear(hidden_size,1)

    def forward(self,x):
        #optional => shape (num of layers, batch size, hidden size)
        h0 = torch.zeros(self.num_layers,x.size(0),self.hidden_size)

        out,_ = self.rnn(x,h0)
        # 1st value = hidden state of all the timesteps
        # 2nd value = final hidden state of last timestep

        out = self.fc(out[:,-1,:])
        return out
        
        

In [28]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

### Training the Model

In [29]:
epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) # add singleton direction
        
        outputs = model(Xb) # (batch_size, 1)

        outputs = torch.sigmoid(outputs.squeeze()) # (batch_size,) => probability

        loss = criterion(outputs, yb) # compute loss
        loss.backward() # backprop
        optimizer.step() # weights update

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")

epoch = 1/10 and loss = 0.34095051884651184
epoch = 2/10 and loss = 0.27884531021118164
epoch = 3/10 and loss = 0.31826648116111755
epoch = 4/10 and loss = 0.26310300827026367
epoch = 5/10 and loss = 0.2871212959289551
epoch = 6/10 and loss = 0.2617632746696472
epoch = 7/10 and loss = 0.2310454547405243
epoch = 8/10 and loss = 0.20525628328323364
epoch = 9/10 and loss = 0.26488107442855835
epoch = 10/10 and loss = 0.36857107281684875


In [30]:
# evaluate

model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0
    
    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy = {correct_vals/tot_vals*100}")

accuracy = 85.34839165070082
